In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

#if you need to mount your code entry point
sys.path.append("native-L2-coordinate-geometry/src")



from config import (
    DATA_PATH,
    MANIFEST_PATH,
    SANDI_CEFR_DICT,
)

from utils import (
    load_pickle,
    save_pickle,
    PhoneClass,
)

from build_phone_codes import (
    AveragingStrategy,
)

from distance_evaluation_sandi import (
    load_distance_pickles,
    analyze_all_distance_associations,
    print_association_results,

)


In [ ]:
all_results = {}


DATASETS_SANDI  = ('train', )
CODEBOOK = "lbs"


MODELS = {
    "wavlm_base_plus": {
        "hf_name": "microsoft/wavlm-base-plus",
        "layer": [6],
    },
    #"wavlm_large": {
    #    "hf_name": "microsoft/wavlm-large",
    #    "layer": [12],#[9, 12, 15],
    #},
    
    #"w2v2_large": {
    #    "hf_name": "facebook/wav2vec2-large-lv60",
    #    "layer": [12], #[12, 15],
    #},
    #"w2v2_xlsr_53": {
    #    "hf_name": "facebook/wav2vec2-large-xlsr-53",
    #    "layer": [12], #[12, 15],
    #},
}



METRICS = [
    "d_euclidean_per_phone",
    "d_mahalanobis_per_phone",
    "d_cosine_per_phone",
]



PHONE_CALSSES = [
    #"monophone",
    #"diphone",
    "triphone",
]

RANKS = [32]

STRATEGIES = [
    "full_unit",
]


DATASETS_SANDI = ('train',)
CODEBOOK = "lbs"

distances_sandi = load_distance_pickles(
    models=MODELS,
    distance_root=f"{DATA_PATH}/distances/{CODEBOOK}_codebook/sandi",
    datasets=DATASETS_SANDI,
    projection_mode='svd',
    
    
)



for encoder in MODELS.keys():
    layers = MODELS[encoder]['layer']
    results = analyze_all_distance_associations(
        distances=distances_sandi,
        encoder=encoder,
        dataset=DATASETS_SANDI[0],
        layers=layers,
        strategies=STRATEGIES,
        phone_classes=PHONE_CALSSES,
        alignments=ALGNS,
        ranks=RANKS,
        metrics=METRICS,
        aggregation="mean",    
        compute_partial=False,
    
        # Keep these False for grid searches.
        bootstrap=False,
        permutation=False,
    )
    all_results[encoder] = results



In [ ]:
best = {}
for k in MODELS.keys():
    print(f'**************Here is encoder: {k}******************')
    print_association_results(all_results[k])